# Round 15: cached-OOF blend scan - can the forest add independent signal? - S6E10

Exp 37 (RandomForest on counts + leak-free TE) measured 0.00237 below the Exp 36 booster, but a
weaker model from a different family can still carry independent signal that a blend exploits.
This round tests that at near-zero compute: no new OOF fits - it reuses the round-12 cache
(Exp 36 OOF) and the round-13 cache (Exp 37/38 OOF), checks integrity by reproducing the logged
fold AUCs, measures OOF correlation, and scans a small fixed weight grid.

Honesty notes, logged up front: (1) blend weights are selected on the same seed-42 OOF the
candidates were tuned on - adaptive selection, bounded to these three tested weights;
(2) the 2x paired-spread band is a decision heuristic, not proof; (3) a blend win is a win for
THIS tested blend, and a loss is bounded to these weights and this pairing - not a verdict on
blending in general. The earlier blend attempt (Exp 25) failed on a highly correlated pair
(corr 0.9943); the forest is a different family, so correlation is the first thing measured.

Guard: the refit fires only if a blend clears the Exp 36 bar (WORKING SIGNAL better under the
paired band). Otherwise no submission file; Exp 36 and submission_round12.csv stand.

In [1]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)
TE_SMOOTH = 100
RF_PARAMS = dict(n_estimators=400, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE)
BOOSTER_BASIS = 249  # round-12 logged refit basis: mean of seed-42 fold-mean best_iterations (275.2 / 222.2)
WEIGHTS = [0.85, 0.90, 0.95]  # booster share; forest share is 1 - w
E36_FOLDS_R12 = [0.96016, 0.95887, 0.96067, 0.96009, 0.96019]       # the bar (seed-42 partition)
E37_FOLDS_R13 = [0.95756, 0.95691, 0.95824, 0.95789, 0.95754]       # integrity reference for the RF cache

In [2]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
COUNT_COLS = list(features.columns)
TE_COLS_ALL = COUNT_COLS

def to_lgbm(df):
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def add_counts(train_part, other_parts=()):
    tables = {c: train_part[c].value_counts() for c in COUNT_COLS}
    def prep(df):
        out = df.copy()
        for c in COUNT_COLS:
            out[c + '_cnt'] = out[c].map(tables[c]).fillna(0).astype(np.int64)
        return out
    return prep(train_part), [prep(p) for p in other_parts]

def te_table(values, labels, m, gm):
    # smoothed value-level target mean: (sum + m*global) / (count + m)
    stats = labels.groupby(values).agg(['sum', 'count'])
    return (stats['sum'] + m * gm) / (stats['count'] + m)

def te_crossfit(X, y, cols, m=TE_SMOOTH, n_inner=5):
    # cross-fitted encodings: a row's own label never enters its encoding (round-10 audit-fixed construction)
    te = pd.DataFrame(index=X.index)
    inner = KFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)
    splits = list(inner.split(np.zeros(len(X))))
    for col in cols:
        enc = np.empty(len(X))
        for itr, iva in splits:
            gm = float(y.iloc[itr].mean())
            t = te_table(X[col].iloc[itr], y.iloc[itr], m, gm)
            enc[iva] = X[col].iloc[iva].map(t).fillna(gm).to_numpy()
        te[col + '_te'] = enc
    return te

def te_apply(X_ref, y_ref, X_new, cols, m=TE_SMOOTH):
    gm = float(y_ref.mean())
    te = pd.DataFrame(index=X_new.index)
    for col in cols:
        t = te_table(X_ref[col], y_ref, m, gm)
        te[col + '_te'] = X_new[col].map(t).fillna(gm).to_numpy()
    return te

def forest_prep(X_tr, y_tr, X_va):
    # counts + leak-free TE, ordinal natives, fold-local median impute (round-13 construction)
    X_tr, (X_va,) = add_counts(X_tr, [X_va])
    te_tr = te_crossfit(X_tr, y_tr, TE_COLS_ALL)
    te_va = te_apply(X_tr, y_tr, X_va, TE_COLS_ALL)
    X_tr = pd.concat([X_tr.reset_index(drop=True), te_tr.reset_index(drop=True)], axis=1)
    X_va = pd.concat([X_va.reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
    for col in CATEGORICAL:
        cats = pd.Categorical(X_tr[col])
        X_tr[col] = cats.codes
        X_va[col] = pd.Categorical(X_va[col], categories=cats.categories).codes
    med = X_tr.median(numeric_only=True)
    return X_tr.fillna(med), X_va.fillna(med)

def fold_aucs(oof, label):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)  # same folds every time
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return mean_d, spread, label

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features')

Full train: 699,635 rows, 21 raw features


## Load caches and verify integrity

The blend arithmetic is only valid if the cached OOFs are exactly the ones the log reports:
the cached Exp 36 and Exp 37 fold AUCs must reproduce the logged constants, and both caches
must hash the same train.csv.

In [3]:
cache_dir = competition_dir / 'oof_cache'
c12 = np.load(cache_dir / 'round12_oof.npz')
c13 = np.load(cache_dir / 'round13_oof.npz')
print('round12 cache keys:', list(c12.keys()))
print('round13 cache keys:', list(c13.keys()))
oof36, oof37, oof38 = c12['exp36'], c13['exp37'], c13['exp38']
y_cache12, y_cache13 = c12['target'], c13['target']
assert np.array_equal(y_cache12, y_cache13), 'target mismatch between caches'
y = pd.Series(y_cache12)

import json as _json
m12 = _json.loads((cache_dir / 'round12_oof.json').read_text())
m13 = _json.loads((cache_dir / 'round13_oof.json').read_text())
print('train sha match:', m12['train_sha256'] == m13['train_sha256'])

f36 = fold_aucs(oof36, y)
f37 = fold_aucs(oof37, y)
d36 = max(abs(a - b) for a, b in zip(f36, E36_FOLDS_R12))
d37 = max(abs(a - b) for a, b in zip(f37, E37_FOLDS_R13))
print(f'cache integrity: exp36 fold-AUC max diff vs logged {d36:.6f} | exp37 {d37:.6f}')
assert d36 < 1e-4 and d37 < 1e-4, 'cache does not reproduce logged fold AUCs - stop'
print('integrity PASS - blends computed on the verified cached OOFs')

round12 cache keys: ['incumbent', 'exp35', 'exp36', 'target']
round13 cache keys: ['incumbent', 'exp37', 'exp38', 'target']
train sha match: True


cache integrity: exp36 fold-AUC max diff vs logged 0.000005 | exp37 0.000004
integrity PASS - blends computed on the verified cached OOFs


## Correlation and blend scan

Correlation first - the Exp 25 blend failed because the pair was 0.9943 correlated. Then the
fixed weight grid, each blend paired against the Exp 36 bar folds.

In [4]:
print(f'OOF correlation exp36 x exp37 (RF):  {np.corrcoef(oof36, oof37)[0, 1]:.4f}')
print(f'OOF correlation exp36 x exp38 (ET):  {np.corrcoef(oof36, oof38)[0, 1]:.4f}  (descriptive only)')
print(f'OOF correlation exp37 x exp38:       {np.corrcoef(oof37, oof38)[0, 1]:.4f}  (descriptive only)')
print()
results = {}
for w in WEIGHTS:
    blend = w * oof36 + (1 - w) * oof37
    fa = fold_aucs(blend, y)
    m_, s_, lab = paired_verdict(fa, E36_FOLDS_R12)
    pooled = roc_auc_score(y, blend)
    results[w] = (fa, m_, lab, pooled)
    print(f'w={w:.2f} booster / {1-w:.2f} forest: folds {[round(a, 5) for a in fa]}')
    print(f'   mean {np.mean(fa):.6f} | pooled {pooled:.6f} | vs Exp 36 bar {m_:+.6f} (spread {s_:.5f}) - {lab}')

OOF correlation exp36 x exp37 (RF):  0.9964
OOF correlation exp36 x exp38 (ET):  0.9949  (descriptive only)
OOF correlation exp37 x exp38:       0.9984  (descriptive only)



w=0.85 booster / 0.15 forest: folds [0.96011, 0.95891, 0.96066, 0.96013, 0.96012]
   mean 0.959986 | pooled 0.959979 | vs Exp 36 bar -0.000010 (spread 0.00004) - NO SIGNAL (within 2x per-fold delta spread - no action)
w=0.90 booster / 0.10 forest: folds [0.96015, 0.95892, 0.96069, 0.96014, 0.96016]
   mean 0.960013 | pooled 0.960008 | vs Exp 36 bar +0.000017 (spread 0.00003) - NO SIGNAL (within 2x per-fold delta spread - no action)


w=0.95 booster / 0.05 forest: folds [0.96017, 0.95891, 0.96069, 0.96013, 0.96019]
   mean 0.960018 | pooled 0.960013 | vs Exp 36 bar +0.000022 (spread 0.00002) - NO SIGNAL (within 2x per-fold delta spread - no action)


In [5]:
winners = [w for w, (fa, m, lab, _) in results.items() if lab.startswith('WORKING SIGNAL: better')]
win = max(winners, key=lambda w: float(np.mean(results[w][0]))) if winners else None
print()
if win:
    print(f'REFIT GUARD: blend w={win:.2f} clears the Exp 36 bar under the paired band -> refit fires '
          f'(exploratory, descriptive signal; weight selected on the same OOF - adaptive, bounded to tested weights).')
else:
    print('REFIT GUARD: no tested blend cleared the Exp 36 bar under the paired band. Exp 36 remains the '
          'preferred candidate; submission_round12.csv is untouched. Bounded to these three weights and this pairing.')


REFIT GUARD: no tested blend cleared the Exp 36 bar under the paired band. Exp 36 remains the preferred candidate; submission_round12.csv is untouched. Bounded to these three weights and this pairing.


## Conditional refit (only if the guard fired)

Booster half: the round-12 refit exactly (counts + TE, 3 seeds x {63, 127} leaves, basis 249).
Forest half: the round-13 refit exactly (forest_prep on full train, RF 400/leaf5, seed 42).
Test predictions blended at the winning weight.

In [6]:
if win is None:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    w = win

    tr_c, (te_c,) = add_counts(features, [test_features])
    te_full = te_crossfit(tr_c, target, TE_COLS_ALL)
    te_test = te_apply(features, target, test_features, TE_COLS_ALL)
    X_full = pd.concat([tr_c.reset_index(drop=True), te_full.reset_index(drop=True)], axis=1)
    X_test = pd.concat([te_c.reset_index(drop=True), te_test.reset_index(drop=True)], axis=1)
    X_full, X_test = to_lgbm(X_full), to_lgbm(X_test)
    preds_booster = []
    for leaves in [63, 127]:
        for seed in [42, 43, 44]:
            cfg = dict(learning_rate=0.05, num_leaves=leaves, n_estimators=BOOSTER_BASIS,
                       n_jobs=-1, random_state=seed, verbose=-1)
            cfg.update(STOCH)
            model = lgb.LGBMClassifier(**cfg)
            model.fit(X_full, target)
            preds_booster.append(model.predict_proba(X_test)[:, 1])
    p_booster = np.mean(preds_booster, axis=0)

    X_full_f, X_test_f = forest_prep(features, target, test_features)
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_full_f, target)
    p_forest = rf.predict_proba(X_test_f)[:, 1]

    p_blend = w * p_booster + (1 - w) * p_forest
    sub = pd.DataFrame({'id': test['id'], 'satisfaction': p_blend})
    sub.to_csv('submission_round15.csv', index=False)
    print(f'submission_round15.csv written: {len(sub):,} rows | blend w={w:.2f}: 6 boosters (basis {BOOSTER_BASIS}) '
          f'+ RF 400/leaf5. Local file only - NOT submitted; Andrea submits.')

No submission file written this round.


In [7]:
# OOF cache (established pattern): persist blend OOFs + metadata for later rounds.
import hashlib, json as _json, time

h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
blends = {f'w{int(w*100)}': w * oof36 + (1 - w) * oof37 for w in WEIGHTS}
np.savez(cache_dir / 'round15_oof.npz', target=y.to_numpy(), **blends)
meta = {
    'round': 15,
    'kind': 'cached-OOF blend scan (no new OOF fits)',
    'train_sha256': h.hexdigest(),
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'weights_booster': WEIGHTS,
    'fold_aucs': {f'w{int(w*100)}': results[w][0] for w in WEIGHTS},
    'guard': {'winner_booster_weight': win, 'bar': 'E36_FOLDS_R12'},
    'adaptive_note': 'weights selected on the same seed-42 OOF the candidates were tuned on',
}
(cache_dir / 'round15_oof.json').write_text(_json.dumps(meta, indent=2))
print('cached oof_cache/round15_oof.npz + round15_oof.json')

cached oof_cache/round15_oof.npz + round15_oof.json


## What happens next

- If a blend cleared the bar: the blend becomes the preferred candidate and
  submission_round15.csv is the new local submission file (Andrea submits); the forest family
  earned its place through blending, not standalone.
- If no blend cleared the bar: the RF preference has now been measured standalone (raw,
  counts, counts+TE) and as a blend partner on the strongest representation - bounded to the
  tested configs and weights, not a proof about the family. Existing-data levers are then
  thinning: remaining options are an honest plateau summary for Andrea, with the external-data
  question presented as his choice (host-linked original dataset, CC0 on its face, provenance
  caveats logged) - not an automatic next step.